# ASRA Phase 1 — Kaggle submission (ARC Prize 2026) - version 11.4

v11-4 fix: **dedicated venv** (`/kaggle/working/asra_venv`, `--without-pip` + `pip --prefix`) with `--system-site-packages` for Kaggle pandas/pyarrow; competition wheels isolated in venv.

## 0. Bootstrap venv + mirror competition assets

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')
COMP_SLUG = 'arc-prize-2026-arc-agi-3'
WORKING = '/kaggle/working' if IS_KAGGLE else '.'
VENV = os.path.join(WORKING, 'asra_venv')
VENV_PY = os.path.join(VENV, 'bin', 'python')
VENV_PIP = os.path.join(VENV, 'bin', 'pip')

if IS_KAGGLE:
    COMP_CANDIDATES = [
        f'/kaggle/input/competitions/{COMP_SLUG}',
        f'/kaggle/input/{COMP_SLUG}',
    ]
    COMP_ROOT = next((p for p in COMP_CANDIDATES if os.path.isdir(p)), COMP_CANDIDATES[0])
else:
    COMP_ROOT = os.environ.get('ASRA_COMP_ROOT', 'private/kaggle-dataset/competition')

AGENTS_ROOT = os.path.join(COMP_ROOT, 'ARC-AGI-3-Agents')
WHEELS_DIR = os.path.join(COMP_ROOT, 'arc_agi_3_wheels')
ENV_DIR = os.path.join(COMP_ROOT, 'environment_files')
RECORDINGS_DIR = os.path.join(WORKING, 'recordings')
MIRROR_ROOT = os.path.join(WORKING, 'asra_competition')
os.makedirs(RECORDINGS_DIR, exist_ok=True)

print('IS_KAGGLE:', IS_KAGGLE)
print('COMP_ROOT:', COMP_ROOT, '| exists:', os.path.isdir(COMP_ROOT))

if IS_KAGGLE and os.path.isdir(COMP_ROOT):
    for name in ('arc_agi_3_wheels', 'ARC-AGI-3-Agents', 'environment_files'):
        src = os.path.join(COMP_ROOT, name)
        dst = os.path.join(MIRROR_ROOT, name)
        if os.path.isdir(src) and not os.path.isdir(dst):
            shutil.copytree(src, dst)
            print('Mirrored', name, '->', dst)
    wheels_mirror = os.path.join(MIRROR_ROOT, 'arc_agi_3_wheels')
    working_wheels = os.path.join(WORKING, 'wheels')
    if os.path.isdir(wheels_mirror) and not os.path.isdir(working_wheels):
        shutil.copytree(wheels_mirror, working_wheels)
        print('Mirrored wheels ->', working_wheels)

if os.path.isdir(AGENTS_ROOT) and AGENTS_ROOT not in sys.path:
    sys.path.insert(0, AGENTS_ROOT)

wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, '*.whl'))) if os.path.isdir(WHEELS_DIR) else []
if wheels:
    if not os.path.isfile(VENV_PY):
        subprocess.check_call(
            [sys.executable, '-m', 'venv', VENV, '--system-site-packages', '--without-pip'],
            stdout=subprocess.DEVNULL,
        )
        print('Created venv ->', VENV)
    py_ver = f'{sys.version_info.major}.{sys.version_info.minor}'
    SITE = os.path.join(VENV, 'lib', f'python{py_ver}', 'site-packages')
    os.makedirs(SITE, exist_ok=True)
    proc = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--target', SITE, '--no-deps', '--upgrade', '-q', *wheels],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError('Venv wheel install failed')
    os.environ['ASRA_VENV'] = VENV
    print('Installed wheels in venv (no-deps):', len(wheels))
    check = subprocess.run(
        [VENV_PY, '-c', 'import arcengine, arc_agi; print("arcengine+arc_agi import OK")'],
        capture_output=True,
        text=True,
    )
    print(check.stdout.strip())
    if check.returncode != 0:
        print(check.stderr)
        raise RuntimeError('Venv import check failed')
else:
    print('WARNING: wheels dir missing')

os.environ.setdefault('RECORDINGS_DIR', RECORDINGS_DIR)
os.environ.setdefault('ENVIRONMENTS_DIR', ENV_DIR if os.path.isdir(ENV_DIR) else 'environment_files')
os.environ['OPERATION_MODE'] = 'COMPETITION' if IS_KAGGLE else os.environ.get('OPERATION_MODE', 'OFFLINE')
os.environ['ASRA_AGENTS_ROOT'] = AGENTS_ROOT
print('OPERATION_MODE:', os.environ['OPERATION_MODE'])

## 1. Write `my_agent.py`

In [ ]:
MY_AGENT_CODE = '''"""ASRA Phase 1 agent for ARC Prize 2026 — ARC-AGI-3 (Kaggle Swarm)."""

from __future__ import annotations

import glob
import hashlib
import json
import os
import random
import subprocess
import sys
import types
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple


def _working_dir() -> str:
    return "/kaggle/working" if os.path.isdir("/kaggle/working") else "."


def _venv_paths(working: str) -> Tuple[str, str, str]:
    venv = os.path.join(working, "asra_venv")
    return venv, os.path.join(venv, "bin", "python"), os.path.join(venv, "bin", "pip")


def _maybe_reexec_venv() -> None:
    """Run inside competition venv (matches notebook bootstrap)."""
    if os.environ.get("ASRA_VENV_ACTIVE") == "1":
        return
    working = _working_dir()
    _, venv_py, _ = _venv_paths(working)
    if not os.path.isfile(venv_py):
        return
    if os.path.realpath(sys.executable) == os.path.realpath(venv_py):
        os.environ["ASRA_VENV_ACTIVE"] = "1"
        return
    env = os.environ.copy()
    env["ASRA_VENV_ACTIVE"] = "1"
    env.setdefault("PYTHONNOUSERSITE", "1")
    os.execve(venv_py, [venv_py, os.path.abspath(__file__), *sys.argv[1:]], env)


_maybe_reexec_venv()


def _is_kaggle_runtime() -> bool:
    if os.path.exists("/kaggle/input"):
        return True
    return os.path.isdir(os.path.join(_working_dir(), "asra_competition"))


def _resolve_comp_root(is_kaggle: bool) -> str:
    if is_kaggle:
        working = _working_dir()
        candidates = [
            "/kaggle/input/competitions/arc-prize-2026-arc-agi-3",
            "/kaggle/input/arc-prize-2026-arc-agi-3",
            os.path.join(working, "asra_competition"),
        ]
        return next((p for p in candidates if os.path.isdir(p)), candidates[0])
    return os.environ.get(
        "ASRA_COMP_ROOT",
        os.path.join("private", "kaggle-dataset", "competition"),
    )


def _verify_runtime(agents_root: str) -> bool:
    if not agents_root or not os.path.isdir(os.path.join(agents_root, "agents")):
        return False
    try:
        import arcengine  # noqa: F401
        import arc_agi  # noqa: F401
    except ImportError:
        return False
    return True


def _venv_site_packages(venv: str) -> str:
    ver = f"{sys.version_info.major}.{sys.version_info.minor}"
    site = os.path.join(venv, "lib", f"python{ver}", "site-packages")
    os.makedirs(site, exist_ok=True)
    return site


def _install_wheels_in_venv(wheels_dir: str, working: str) -> str:
    """Create venv without ensurepip; install wheels into venv site-packages only."""
    venv, venv_py, _ = _venv_paths(working)
    if not os.path.isfile(venv_py):
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "venv",
                venv,
                "--system-site-packages",
                "--without-pip",
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
        )
    wheels = sorted(glob.glob(os.path.join(wheels_dir, "*.whl")))
    if wheels:
        target = _venv_site_packages(venv)
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--target", target, "--no-deps", "--upgrade", "-q", *wheels],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"Venv wheel install failed:\\n{result.stderr}")
    os.environ["ASRA_VENV"] = venv
    return venv_py


def _bootstrap_kaggle() -> str:
    """Install competition wheels into venv and wire ARC-AGI-3-Agents."""
    is_kaggle = _is_kaggle_runtime()
    working = _working_dir()
    comp_root = _resolve_comp_root(is_kaggle)
    agents_root = os.path.join(comp_root, "ARC-AGI-3-Agents")
    wheels_dir = os.path.join(comp_root, "arc_agi_3_wheels")
    working_wheels = os.path.join(working, "wheels")
    if not os.path.isdir(wheels_dir) and os.path.isdir(working_wheels):
        wheels_dir = working_wheels

    env_dir = os.path.join(comp_root, "environment_files")
    working_env = os.path.join(working, "environment_files")
    if not os.path.isdir(env_dir) and os.path.isdir(working_env):
        env_dir = working_env

    recordings_dir = os.path.join(working, "recordings")
    os.makedirs(recordings_dir, exist_ok=True)

    if os.path.isdir(agents_root) and agents_root not in sys.path:
        sys.path.insert(0, agents_root)

    os.environ.setdefault("RECORDINGS_DIR", recordings_dir)
    os.environ.setdefault(
        "ENVIRONMENTS_DIR",
        env_dir if os.path.isdir(env_dir) else "environment_files",
    )
    os.environ["OPERATION_MODE"] = (
        "COMPETITION" if is_kaggle else os.environ.get("OPERATION_MODE", "OFFLINE")
    )
    os.environ["ASRA_AGENTS_ROOT"] = agents_root

    if _verify_runtime(agents_root):
        return agents_root

    if os.path.isdir(wheels_dir):
        venv_py = _install_wheels_in_venv(wheels_dir, working)
        if os.path.realpath(sys.executable) != os.path.realpath(venv_py):
            env = os.environ.copy()
            env["ASRA_VENV_ACTIVE"] = "1"
            env.setdefault("PYTHONNOUSERSITE", "1")
            os.execve(venv_py, [venv_py, os.path.abspath(__file__), *sys.argv[1:]], env)

    if os.path.isdir(agents_root) and agents_root not in sys.path:
        sys.path.insert(0, agents_root)

    if not _verify_runtime(agents_root):
        raise RuntimeError(
            "ARC-AGI-3 runtime not ready after venv bootstrap. "
            f"comp_root={comp_root!r} agents_root={agents_root!r} wheels_dir={wheels_dir!r}"
        )
    return agents_root


def _load_agents_runtime(agents_root: str):
    """Load Agent + Swarm without agents/__init__.py (avoids optional langgraph templates)."""
    import importlib.util

    agents_dir = os.path.join(agents_root, "agents")
    if not os.path.isdir(agents_dir):
        raise RuntimeError(f"agents package not found under {agents_root}")

    if agents_root not in sys.path:
        sys.path.insert(0, agents_root)

    pkg = sys.modules.get("agents")
    if pkg is None or not getattr(pkg, "_ASRA_STUB", False):
        pkg = types.ModuleType("agents")
        pkg.__path__ = [agents_dir]
        pkg.__package__ = "agents"
        pkg.AVAILABLE_AGENTS = {}
        pkg._ASRA_STUB = True
        sys.modules["agents"] = pkg

    def _load_submodule(name: str):
        fq = f"agents.{name}"
        if fq in sys.modules:
            return sys.modules[fq]
        path = os.path.join(agents_dir, f"{name}.py")
        spec = importlib.util.spec_from_file_location(fq, path)
        if spec is None or spec.loader is None:
            raise ImportError(f"Cannot load {fq} from {path}")
        mod = importlib.util.module_from_spec(spec)
        mod.__package__ = "agents"
        sys.modules[fq] = mod
        spec.loader.exec_module(mod)
        return mod

    _load_submodule("tracing")
    _load_submodule("recorder")
    agent_mod = _load_submodule("agent")
    swarm_mod = _load_submodule("swarm")
    return agent_mod.Agent, swarm_mod.Swarm, pkg.AVAILABLE_AGENTS


AGENTS_ROOT = _bootstrap_kaggle()

import numpy as np
import pandas as pd
from arcengine import FrameData, GameAction, GameState

Agent, Swarm, AVAILABLE_AGENTS = _load_agents_runtime(AGENTS_ROOT)

SEED = int(os.environ.get("ASRA_SEED", "42"))
MAX_ACTIONS = int(os.environ.get("ASRA_MAX_ACTIONS", "80"))
SIMPLE_ACTIONS = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]

random.seed(SEED)
np.random.seed(SEED)


def canonical_grid(grid: Any) -> List[List[int]]:
    arr = np.array(grid, dtype=int)
    return arr.tolist()


def state_hash(grid: Any) -> str:
    payload = json.dumps(canonical_grid(grid), separators=(",", ":"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


class ActionSemanticsInferencer:
    def __init__(self) -> None:
        self.effects: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)

    def observe(self, state_hash_value: str, action: str, diff: Dict[str, Any], reward: float) -> None:
        self.effects[(state_hash_value, action)].append(
            {"num_changed_cells": diff.get("num_changed_cells"), "reward": reward}
        )

    def infer(self, state_hash_value: str, action: str) -> Dict[str, Any]:
        effects = self.effects.get((state_hash_value, action), [])
        if not effects:
            return {"observations": 0, "hypothesis": "unknown", "consistency_score": None}
        counts = [e["num_changed_cells"] for e in effects if e["num_changed_cells"] is not None]
        std = float(np.std(counts)) if counts else 0.0
        mean = float(np.mean(counts)) if counts else None
        if mean == 0:
            hyp = "no-op / blocked"
        elif mean is not None and mean <= 1.5:
            hyp = "localized cell update"
        else:
            hyp = "multi-cell transform"
        return {
            "observations": len(effects),
            "hypothesis": hyp,
            "consistency_score": float(1.0 / (1.0 + std)) if counts else None,
        }


class ASRAExplorer:
    def __init__(self, action_names: List[str]) -> None:
        self.action_names = action_names
        self.state_action_counts: Counter = Counter()
        self.action_rewards: Dict[str, List[float]] = defaultdict(list)
        self.dead_ends: set = set()

    def update(self, state_hash_value: str, action: str, diff: Dict[str, Any], reward: float) -> None:
        self.state_action_counts[(state_hash_value, action)] += 1
        self.action_rewards[action].append(float(reward))
        if diff.get("num_changed_cells") == 0 and reward <= 0:
            self.dead_ends.add((state_hash_value, action))

    def choose_action(
        self,
        state_hash_value: str,
        semantics: ActionSemanticsInferencer,
        available: Optional[List[str]] = None,
    ) -> str:
        candidates = [a for a in self.action_names if available is None or a in available] or list(
            self.action_names
        )
        scores: Dict[str, float] = {}
        for action in candidates:
            if (state_hash_value, action) in self.dead_ends:
                continue
            sem = semantics.infer(state_hash_value, action)
            c = sem.get("consistency_score")
            uncertainty = 1.0 if c is None else (1.0 - min(1.0, c))
            local = self.state_action_counts[(state_hash_value, action)]
            mean_r = float(np.mean(self.action_rewards[action])) if self.action_rewards[action] else 0.0
            scores[action] = 2.0 / (1.0 + local) + 0.7 * uncertainty + 0.5 * mean_r + random.random() * 0.05
        return max(scores.items(), key=lambda kv: kv[1])[0] if scores else random.choice(candidates)


GLOBAL_SEMANTICS = ActionSemanticsInferencer()
GLOBAL_EXPLORER = ASRAExplorer(SIMPLE_ACTIONS)


class ASRAAgent(Agent):
    """Transition-centric explorer with state-conditioned action semantics."""

    MAX_ACTIONS = MAX_ACTIONS

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS

    def _available_simple(self, latest_frame: FrameData) -> List[str]:
        avail = getattr(latest_frame, "available_actions", None) or []
        names = [a.name for a in avail if hasattr(a, "name") and a.name in SIMPLE_ACTIONS]
        return names or SIMPLE_ACTIONS

    def _to_game_action(self, action_name: str, grid: Any) -> GameAction:
        ga = getattr(GameAction, action_name)
        if ga.is_complex():
            h, w = len(grid), len(grid[0]) if grid else 0
            ga.set_data({"x": w // 2, "y": h // 2})
        ga.reasoning = f"ASRA v0.3: {action_name}"
        return ga

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        grid = latest_frame.frame
        name = GLOBAL_EXPLORER.choose_action(
            state_hash(grid), GLOBAL_SEMANTICS, self._available_simple(latest_frame)
        )
        return self._to_game_action(name, grid)

    def take_action(self, action: GameAction) -> Optional[FrameData]:
        self._last_action_name = action.name
        return super().take_action(action)

    def append_frame(self, frame: FrameData) -> None:
        super().append_frame(frame)
        if len(self.frames) < 2:
            return
        prev, curr = self.frames[-2], self.frames[-1]
        if not prev.frame or not curr.frame:
            return
        diff_count = int(np.sum(np.array(prev.frame) != np.array(curr.frame)))
        diff = {"num_changed_cells": diff_count}
        reward = float(getattr(curr, "levels_completed", 0) or 0)
        sh = state_hash(prev.frame)
        action = getattr(self, "_last_action_name", "UNKNOWN")
        GLOBAL_SEMANTICS.observe(sh, action, diff, reward)
        GLOBAL_EXPLORER.update(sh, action, diff, reward)


def _game_ids(environments: Any) -> List[str]:
    ids: List[str] = []
    for env in environments:
        if isinstance(env, str):
            ids.append(env)
        elif hasattr(env, "game_id"):
            ids.append(str(env.game_id))
        else:
            ids.append(str(env))
    return ids


def scorecard_to_rows(scorecard: Any) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if scorecard is None:
        return rows
    environments = getattr(scorecard, "environments", None) or []
    for env in environments:
        env_id = getattr(env, "id", None) or getattr(env, "game_id", "unknown")
        runs = getattr(env, "runs", None) or []
        if not runs:
            rows.append(
                {
                    "game_id": env_id,
                    "score": 0.0,
                    "levels_completed": 0,
                    "actions": 0,
                    "completed": False,
                }
            )
            continue
        best = max(runs, key=lambda r: (getattr(r, "score", 0.0), getattr(r, "levels_completed", 0)))
        rows.append(
            {
                "game_id": env_id,
                "score": float(getattr(best, "score", 0.0) or 0.0),
                "levels_completed": int(getattr(best, "levels_completed", 0) or 0),
                "actions": int(getattr(best, "actions", 0) or 0),
                "completed": bool(getattr(best, "completed", False)),
            }
        )
    return rows


def write_submission_parquet(path: str, scorecard: Optional[Any] = None) -> None:
    rows = scorecard_to_rows(scorecard)
    if not rows:
        rows = [
            {
                "game_id": "placeholder",
                "score": 0.0,
                "levels_completed": 0,
                "actions": 0,
                "completed": False,
            }
        ]
    pd.DataFrame(rows).to_parquet(path, index=False)


def run_swarm() -> Any:
    """Play all competition games via official Swarm (called on Kaggle re-run)."""
    from arc_agi import Arcade

    agents_pkg = sys.modules.get("agents")
    if agents_pkg is not None:
        agents_pkg.AVAILABLE_AGENTS = AVAILABLE_AGENTS
    AVAILABLE_AGENTS["asra"] = ASRAAgent

    arcade = Arcade()
    games = _game_ids(arcade.get_environments())
    print(f"ASRA Swarm: {len(games)} games | OPERATION_MODE={os.environ.get('OPERATION_MODE')}")
    swarm = Swarm(
        "asra",
        os.environ.get("ARC_BASE_URL", "https://three.arcprize.org"),
        games,
        tags=["asra-v0.3"],
    )
    return swarm.main()


def self_test() -> None:
    """Validate scoring imports without running full Swarm."""
    print(f"python={sys.executable}")
    print(f"OPERATION_MODE={os.environ.get('OPERATION_MODE')}")
    assert hasattr(ASRAAgent, "choose_action")
    assert callable(run_swarm)
    print("self-test OK")


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "--self-test":
        self_test()
        sys.exit(0)

    working = _working_dir()
    submission_path = os.path.join(working, "submission.parquet")
    try:
        scorecard = run_swarm()
        write_submission_parquet(submission_path, scorecard)
        if scorecard is not None:
            print("Scorecard score:", getattr(scorecard, "score", None))
        print("Wrote", submission_path)
    except Exception:
        import traceback

        traceback.print_exc()
        raise
'''

MY_AGENT_PATH = os.path.join(WORKING, 'my_agent.py')
with open(MY_AGENT_PATH, 'w', encoding='utf-8') as f:
    f.write(MY_AGENT_CODE)
print('Wrote', MY_AGENT_PATH, '| bytes:', os.path.getsize(MY_AGENT_PATH))

## 2. Smoke-test (venv python — simulates scoring)

In [ ]:
import subprocess

result = subprocess.run(
    [VENV_PY, MY_AGENT_PATH, '--self-test'],
    cwd=WORKING,
    env={**os.environ, 'ASRA_VENV_ACTIVE': '1', 'PYTHONNOUSERSITE': '1'},
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('my_agent.py venv self-test failed (scoring simulation)')
print('my_agent.py venv self-test OK (scoring simulation)')

## 3. Write `submission.parquet` (Kaggle submit gate)

In [ ]:
import subprocess

SUBMISSION_PATH = os.path.join(WORKING, 'submission.parquet')
gate_code = (
    'import os, pandas as pd; '
    f'p={SUBMISSION_PATH!r}; '
    'pd.DataFrame([{"game_id":"placeholder","score":0.0,"levels_completed":0,"actions":0,"completed":False}]).to_parquet(p, index=False); '
    'print("Wrote", p, "| bytes:", os.path.getsize(p))'
)
gate = subprocess.run([VENV_PY, '-c', gate_code], capture_output=True, text=True)
print(gate.stdout.strip())
if gate.returncode != 0:
    print(gate.stderr)
    raise RuntimeError('submission.parquet write failed in venv')
assert os.path.isfile(SUBMISSION_PATH), 'submission.parquet missing'
print('Output files:', sorted(os.listdir(WORKING)))

## Done

Competition runtime in `asra_venv/`. Scoring re-execs `my_agent.py` via venv python.